# **Port Charles stress test case - Influence of input parameters**

We are now going to toy around with some of the inputs. We will focus on the influence of
- Adjusting the simulation region
- Boundary conditions (river injections, tide...)
- Grid resolution
> See parameters section of the manual for details https://cyprienbosserelle.github.io/BG_Flood/ParametersList-py/
---


### **1. Adjusting the region of interest**
Looking closely at the h/h_P0 (timestep/Band=6), unphysical stripes are noticeable. This is caused by the domain extending beyond the range of the DEM (open `PortCharles_DEM.nc` to check)

By default, the simulation domain adjusts with the dimensions of the DEM but misalignments can appear (such as here)
To bypass this issue, the shape of the domain can be adjusted in multiple ways.

#### *1.1 Specify domain bounds*
The domain bounds can be set directly in the BG_param.txt file. 

```markdown
################
# Domain 
################
xmin = 1815750.
xmax = 1824850.
ymin = 5950380.
ymax = 5958440.
```

#### *1.2 Area of interest polygon*
The area of interest (AOI) polygon is a mask which forces BG-Flood to ignore the cells outside the polygon defined by the mask. The AOI does not modify the shape of the domain but reduces the computational load by the number of cells masked. It is usually used in conjunction with defining the domain bounds 

It must be 2 column comma-separated table *.txt file containing the x,y coordinates of the serie of points defining the polygon mask.
> Note: For the polygon to be valid, the first point must be repeated at the end.

```markdown
################
# Domain 
################
xmin = 1815750.
xmax = 1824850.
ymin = 5950380.
ymax = 5958440.

aoi = aoi_square.txt
```

***-> Try to create a simple AOI polygon and compare the simulation (runtime and output) with `aoi_watershed.txt`***



### **2. Boundary conditions**
4 Main types of boundary conditions are available.

```markdown
################
# Boundaries
################
# Main types of boundaries depending on the flag
# 0: Wall
# 1: Zero-gradient (Neumann): Used for open outlets
# 2: Fixed value (Dirichlet) - Requires timeseries file: Used to impose tide
# 3: Absorbing wave - Same as 2 but more stable numerically (allows waves to exit the domain)
left = 0;
right = tide.txt,2; #0;
top = tide.txt,2; #0;
bot = 0;
```

***Try to switch the top and right boundary conditions to have them vary with tide.txt. What do you notice?***
 - Try to check the tide file and vary the amplitude and frequency.
 - Do you need to increase the simulation time?

### **4. Checking the results**

#### 1. In QGIS
To quickly check the output, simply drag and drop the output file into QGIS.

Try to open the results ***Port_Charles_dx32m.nc*** and read the variable ***h_P0***.

> The output variables have a suffix *_P0. We will explain its origin later

#### 2. In Python
The conda environment should already be installed at this stage. If not, refer to section 3 of the *BG_Flood_introduction.ipynb*.

We will use the packages xarray and rioxarray to open the files and plot as follow

In [20]:
#%%% IMPORT PACKAGES
import xarray as xr
import rioxarray
import numpy as np
#import folium


In [ ]:
#%%% SIMPLE PLOT



# Replace with the path to your output file.
BGout_path = "/mnt/e/04_BG-Flood/Tutorial/Tutorial_3_Simple_Rain_Port_Charles/Port_Charles_dx32m.nc"
#BGout_path = "\path\to\Tutorial_1\Port_Charles_dx32m.nc"

# Open the dataset with xarray.
ds = xr.open_dataset(BGout_path)

# Simple plot
timestep = 4
ds["h_P0"][timestep].plot(vmin=0, vmax=2, cmap="Blues")


In [ ]:
# Try to add OpenStreetMap as background
import contextily as ctx
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(14, 10))

timestep = 4
ds["h_P0"][timestep].plot(ax=ax, vmin=0, vmax=2, cmap="Blues")

ctx.add_basemap(
    ax,
)

In [42]:
#%%% Simple plot

# Function to read in the output file and concatenate variables across levels.
def prep_BGout(file_path, list_variables=["h"], crs="EPSG:2193"):
    print(f" # Reading input file {file_path}...")
    with xr.open_dataset(file_path) as ds:
        for var in list_variables:
            print(f"    . Treatment of variable {var}...")
            lvl_str = f"P0"
            da_lvl = ds[f"{var}_{lvl_str}"]
            da_lvl = da_lvl.rename({
                f"yy_{lvl_str}": "y",
                f"xx_{lvl_str}": "x",
            })
            da_lvl = da_lvl.rename(var)
            da_lvl_out = da_lvl.copy()
            y = da_lvl_out.y
            x = da_lvl_out.x

            da_lvl_out = da_lvl_out.rio.write_crs(crs, inplace=True)

        dsout[var] = da_lvl_out

    dsout.rio.write_crs(crs, inplace=True)
    dsout.rio.write_coordinate_system(inplace=True)

    return dsout


# Replace with the path to your output file.
BGout_path = "/mnt/e/04_BG-Flood/Tutorial/Tutorial_3_Simple_Rain_Port_Charles/Port_Charles_dx32mTo8m_Kurganov.nc"
#BGout_path = "\path\to\Tutorial_1\Port_Charles_dx32m.nc"
save_flag = False

# List of variables to concatenate across levels.
list_variables = ["h", "u", "v", "zs", "hmax", "zsmax", "Umax"]

# Initialise output file
dsout = xr.Dataset()

# Rename variables and dimensions to remove 

dsout = prep_BGout(BGout_path, list_variables, crs="EPSG:2193")
# Save file
if save_flag:
    outfile_path = "/mnt/e/04_BG-Flood/Tutorial/Tutorial_3_Simple_Rain_Port_Charles/Port_Charles_dx32mTo8m_Kurganov_concat.nc"
    dsout.to_netcdf(outfile_path)


 # Reading input file /mnt/e/04_BG-Flood/Tutorial/Tutorial_3_Simple_Rain_Port_Charles/Port_Charles_dx32mTo8m_Kurganov.nc...
    . Treatment of variable h...
    . Treatment of variable u...
    . Treatment of variable v...
    . Treatment of variable zs...
    . Treatment of variable hmax...
    . Treatment of variable zsmax...
    . Treatment of variable Umax...


In [ ]:
#%%% Merge outputs from different refinement levels.
# The output file is written in the same format as the input file, but with the variables concatenated across levels.
# The output file can be used for further analysis or visualization.

# Function to read and merge BG-Flood output variables variables across levels.
def merge_levels(file_path, list_variables=["h"], list_lvl=[0,1,2], crs="EPSG:2193"):
    # Initialise output file
    dsout = xr.Dataset()
    with xr.open_dataset(BGout_path) as ds:
        for var in list_variables:
            print(f"    . Treatment of variable {var}...")
            for lvl in list_lvl:
                lvl_str = f"P{lvl}"
                da_lvl = ds[f"{var}_{lvl_str}"]
                da_lvl = da_lvl.rename({
                    f"yy_{lvl_str}": "y",
                    f"xx_{lvl_str}": "x",
                })
                da_lvl = da_lvl.rename(var)
                if lvl == maxlevel:
                    da_lvl_out = da_lvl.copy()
                    y = da_lvl_out.y
                    x = da_lvl_out.x
                    continue

                da_lvl = da_lvl.interp(
                    y=y,
                    x=x,
                    method="nearest"
                )

                da_lvl_out = xr.where(
                    da_lvl_out.isnull() & da_lvl.isnull(),
                    np.nan,
                    da_lvl_out.fillna(0) + da_lvl.fillna(0)
                )
                da_lvl_out = da_lvl_out.rio.write_crs("EPSG:2193", inplace=True)

            dsout[var] = da_lvl_out

    dsout.rio.write_crs("EPSG:2193", inplace=True)
    dsout.rio.write_coordinate_system(inplace=True)

    return dsout

BGout_path = "/mnt/e/04_BG-Flood/Tutorial/Tutorial_3_Simple_Rain_Port_Charles/Port_Charles_dx32mTo8m_Kurganov.nc"

# Min and max refinement levels to concatenate.
minlevel = 0
maxlevel = 2

list_lvl = list(range(maxlevel, minlevel-1, -1))

# List of variables to concatenate across levels.
list_variables = ["h", "u", "v", "zs", "hmax", "zsmax", "Umax"]

dsout = merge_levels(BGout_path, list_variables=list_variables, list_lvl=list_lvl, crs="EPSG:2193")
print(" # Reading input file...")


outfile_path = "/mnt/e/04_BG-Flood/Tutorial/Tutorial_3_Simple_Rain_Port_Charles/Port_Charles_dx32mTo8m_Kurganov_concat.nc"
dsout.to_netcdf(outfile_path)


 # Reading input file...
    . Treatment of variable h...
    . Treatment of variable u...
    . Treatment of variable v...
    . Treatment of variable zs...
    . Treatment of variable hmax...
    . Treatment of variable zsmax...
    . Treatment of variable Umax...


In [36]:
# Sets the level of compression of the file. Try to save with and without compression to see the difference in file size and read/write speed.
encoding = {
    var: {
        "zlib": True,
        "complevel": 6
    }
    for var in dsout.data_vars
}

outfile_path = "/mnt/e/04_BG-Flood/Tutorial/Tutorial_3_Simple_Rain_Port_Charles/Port_Charles_dx32mTo8m_Kurganov_concat_compress.nc"
dsout.to_netcdf(outfile_path, encoding=encoding)